<a href="https://colab.research.google.com/github/Ayoraham/receipt_scanner_V2/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pytesseract

In [ ]:
!pip install "Pillow>=10.0.0,<11.0.0" --upgrade

import PIL
print(f"Verified Pillow Version: {PIL.__version__}")

In [ ]:
import zipfile
from pathlib import Path
src_img_pth = "/content/drive/MyDrive/processed_data.zip"
dest_pth = "data"
with zipfile.ZipFile(src_img_pth,'r') as im_pth:
  im_pth.extractall(dest_pth)
print(f"All files extracted to {dest_pth}")

In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification

In [ ]:
from pathlib import Path
import os

In [ ]:
data_pth = Path('data/content/screenshot_data')

In [ ]:
from google.colab.patches import cv2_imshow as show
img_pths = list(data_pth.glob("*/*"))
show(cv.imread(img_pths[0]))

In [ ]:
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base")
processor.image_processor.apply_ocr = False

In [ ]:
conj = []
for sub in data_pth.iterdir():
  for subsub in sub.iterdir():
    conj.append(subsub)

### Quick Test

In [ ]:
import cv2 as cv
processor.image_processor.apply_ocr = True
img = cv.imread(conj[50])

In [ ]:
encoding = processor(img,
                     max_length=512,
                     truncation=True,
                     padding="max_length",
                     return_tensors='pt')

In [ ]:
model = LayoutLMv3ForSequenceClassification.from_pretrained('microsoft/layoutlmv3-base')

In [ ]:
output = model(**encoding)


In [ ]:
output

In [ ]:
!pip install -qqq torchmetrics
!pip install -qqq pytorch_lightning

In [ ]:
from torchmetrics import Accuracy
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
import json


In [ ]:
et_loc = "/content/drive/MyDrive/layoutlm_dataset.json"
with open(et_loc,'r') as file:
  extracted = json.load(file)

In [ ]:
!gdown 1bQ4mFbVRUtOEJSe8b4hUYIcngSgfdldw

In [ ]:
!tar -xf financial-documents-ocr.tar.xz

In [ ]:
from pathlib import Path
img_pths = sorted(list(Path("images").glob("*/*.jpg")))
print(len(img_pths))

#Dataset

In [ ]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

In [ ]:
img_pths[3]

In [ ]:
def scale_bbox(bbox:list[int],width_scale:[float],height_scale:[float]) -> list[int]:
  return[
      int(bbox[0]*width_scale),
      int(bbox[1]*height_scale),
      int(bbox[2]*width_scale),
      int(bbox[3]*height_scale)
  ]

In [ ]:
doc_classes = [p.name for p in Path("images").glob("*")]
doc_classes

In [ ]:
processor.image_processor.apply_ocr = False

In [ ]:
from torch.utils.data import Dataset,DataLoader
from PIL import Image
import torch

# The dataset class is what will be passed into the model for training, its also for ease, cuz its wraps your whole dataset in a pytorch dataset class

class DocunmentClassificationDataset(Dataset):

  def __init__(self,image_paths,processor):
    self.image_paths = image_paths
    self.processor = processor

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self,item):
    image_path = self.image_paths[item]
    image = Image.open(image_path).convert("RGB")
    width,height = image.size

    json_path = image_path.with_suffix(".json")

    with json_path.open("r") as f:
      ocr_result = json.load(f)

    width_scale,height_scale = 1000/width, 1000/height

    words = []
    boxes = []
    for row in ocr_result:
      words.append(row['word'])
      boxes.append(scale_bbox(row['bounding_box'],width_scale,height_scale))
    label = doc_classes.index(image_path.parent.name)

    encoding = processor(
        image,
        words,
        boxes=boxes,
        max_length=512,
        truncation=True,
        return_tensors='pt',
        padding="max_length"
    )
    encoding = {k: v.squeeze() for k,v in encoding.items()}
    encoding['labels'] = torch.tensor(label)

    return encoding


In [ ]:
train_images, test_images = train_test_split(img_pths, test_size=0.2)
len(train_images),len(test_images)

In [ ]:
train_dataset = DocunmentClassificationDataset(train_images,processor)
test_dataset = DocunmentClassificationDataset(test_images,processor)

In [ ]:
for item in train_dataset:
  print(item['bbox'].shape)
  print(item['labels'])
  break

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2
)



In [ ]:
import pytorch_lightning as pl
class ModelModule(pl.LightningModule):

  def __init__(self, n_classes: int):
    super().__init__()
    self.model = LayoutLMv3ForSequenceClassification.from_pretrained(
        "microsoft/layoutlmv3-base",
        num_labels=n_classes
    )
    self.train_accuracy = Accuracy(task="multiclass",num_classes=n_classes)
    self.val_accuracy = Accuracy(task="multiclass",num_classes=n_classes)

  def on_train_start(self):
    self.model.train()

  def forward(self, input_ids,attention_mask, bbox, pixel_values, labels=None):
    return self.model(
        input_ids,
        attention_mask=attention_mask,
        bbox=bbox,
        pixel_values=pixel_values,
        labels=labels
    )

  def training_step(self,batch,batch_idx):
    labels = batch['labels']
    outputs = self(
        batch["input_ids"],
        batch['attention_mask'],
        batch['bbox'],
        batch['pixel_values'],
        labels
    )
    loss = outputs.loss
    self.log("train_loss",loss)
    self.train_accuracy(outputs.logits,labels)
    self.log("train_acc",self.train_accuracy, on_step=True, on_epoch=True)
    return loss

  def validation_step(self,batch,batch_idx):
    labels = batch['labels']
    outputs = self(
        batch["input_ids"],
        batch['attention_mask'],
        batch['bbox'],
        batch['pixel_values'],
        labels
    )
    loss = outputs.loss
    self.log("val_loss", loss, prog_bar=True)
    self.val_accuracy(outputs.logits,labels)
    self.log("val_acc",self.val_accuracy, on_step=False, on_epoch=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.Adam(self.model.parameters(), lr=0.00001)

In [ ]:
model_module = ModelModule(len(doc_classes))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

In [ ]:
model_checkpoint = ModelCheckpoint(
    filename= "{epoch}-{step}-{val_loss:.4f}",
    save_last= True,
    save_top_k=3,
    monitor="val_loss",
    mode="min"
)

trainer = pl.Trainer(
    accelerator="gpu",
    precision=16,
    devices=1,
    max_epochs=4,
    callbacks=[
        model_checkpoint
    ],
    default_root_dir="./checkpoints"
)

In [ ]:
trainer.fit(model_module, train_dataloader, test_dataloader)